In [50]:
# Import the necessary libraries

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import BertTokenizer, BertModel
from category_encoders import BinaryEncoder, OneHotEncoder, TargetEncoder
from pathlib import Path
from tqdm import tqdm
import torch


In [32]:
# A class to load the data from the json files

class DataLoader:
    def __init__(self, data_path):
        self.data_path = data_path

    def load_data(self):
        data_list = []

        for file in os.listdir(self.data_path):
            if file.endswith('.json'):
                data = self._extract_data(os.path.join(self.data_path, file))
                if all(value != '' for value in data.values()):
                    data_list.append(data)

        return pd.DataFrame(data_list)

    def _extract_data(self, file_path):
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)

        protocol = data.get('protocolSection', {})
        identification = protocol.get('identificationModule', {})
        description = protocol.get('descriptionModule', {})
        conditions = protocol.get('conditionsModule', {})
        design = protocol.get('designModule', {})
        eligibility = protocol.get('eligibilityModule', {})

        primary_purpose = design.get('designInfo', {}).get('primaryPurpose', '')


        extracted_data = {
            "NCT ID": self._try_except(identification, 'nctId'),
            "Study Type": self._try_except(design, 'studyType'),
            "Standard Ages": self._try_except(eligibility, 'stdAges', join=True),
            "Conditions": self._try_except(conditions, 'conditions', join=True),
            "Eligibility Criteria": self._try_except(eligibility, 'eligibilityCriteria'),
            "Sex": self._try_except(eligibility, 'sex')
        }

        return extracted_data

    def _try_except(self, data_dict, field_name, join=False):
        try:
            value = data_dict.get(field_name, '')
            if join and isinstance(value, list):
                return ", ".join(value)
            return value
        except Exception as e:
            print(f"Error extracting {field_name}: {str(e)}")
            return ''

In [33]:
def extract_raw_data(input_dir=None, output_file=None):
    '''
    Extracts raw data from the json files and saves it to a csv file.

    Parameters:
    - input_dir (str): The directory containing the input JSON files.
    - output_file (str): The name of the CSV file to save the raw data.

    Returns:
    None
    '''
    if input_dir is None or output_file is None:
        raise ValueError("Both 'input_dir' and 'output_file' must be specified.")

    # Assuming DataLoader is a class that loads data from JSON files
    dataloader = DataLoader(input_dir)
    raw_data = dataloader.load_data()

    # Save raw data to CSV
    raw_data.to_csv(output_file, index=False)

    print(f"Raw data extracted to {output_file}")


In [34]:
input_dir = Path(r'./ctg-studies.json')
output_file = Path(r'./ctg-studies.csv')

extract_raw_data(input_dir=input_dir, output_file=output_file)

Raw data extracted to ctg-studies.csv


In [39]:
raw_data = pd.read_csv(r'.\ctg-studies.csv')
raw_data.isna().sum()

NCT ID                  0
Study Type              0
Standard Ages           0
Conditions              0
Eligibility Criteria    0
Sex                     0
dtype: int64

In [52]:
raw_data.describe()

,NCT ID,Study Type,Standard Ages,Conditions,Eligibility Criteria,Sex
count,1266,1266,1266,1266,1266,1266
unique,1266,3,6,862,1254,3
top,NCT00019136,INTERVENTIONAL,"ADULT, OLDER_ADULT",Multiple Myeloma,"Inclusion Criteria:\n\n1. Age ≥18 years old, m...",ALL
freq,1,1198,836,78,3,1239


In [44]:
raw_data.head()

,NCT ID,Study Type,Standard Ages,Conditions,Eligibility Criteria,Sex
0,NCT00019136,INTERVENTIONAL,"ADULT, OLDER_ADULT",Ovarian Cancer,DISEASE CHARACTERISTICS:\n\n* Histologically p...,FEMALE
1,NCT00085930,INTERVENTIONAL,"CHILD, ADULT",Neuroblastoma,Inclusion Criteria:\n\nHigh risk neuroblastoma...,ALL
2,NCT00881920,INTERVENTIONAL,"ADULT, OLDER_ADULT","Lymphoma, Myeloma, Leukemia",INCLUSION CRITERIA:\n\nBLOOD PROCUREMENT:\n\n*...,ALL
3,NCT00902044,INTERVENTIONAL,"CHILD, ADULT, OLDER_ADULT",Sarcoma,INCLUSION CRITERIA:\n\nProcurement Eligibility...,ALL
4,NCT00924326,INTERVENTIONAL,"ADULT, OLDER_ADULT","Primary Mediastinal B-cell Lymphoma, Diffuse, ...",* INCLUSION CRITERIA:\n\n 1. Patient must hav...,ALL


In [45]:
raw_data['Study Type'].value_counts()

Study Type
INTERVENTIONAL     1198
OBSERVATIONAL        63
EXPANDED_ACCESS       5
Name: count, dtype: int64

In [46]:
raw_data['Standard Ages'].value_counts()

Standard Ages
ADULT, OLDER_ADULT           836
CHILD, ADULT, OLDER_ADULT    316
CHILD, ADULT                  97
ADULT                          9
CHILD                          7
OLDER_ADULT                    1
Name: count, dtype: int64

In [47]:
raw_data['Sex'].value_counts()

Sex
ALL       1239
FEMALE      15
MALE        12
Name: count, dtype: int64

In [74]:
def extract_eligibility_features(data=None, bert_model=None, tokenizer=None):
    '''
    Extracts features from the eligibility criteria using the BERT model.

    Parameters:
    - data (pd.DataFrame): The data containing the eligibility criteria.
    - bert_model (str): The name of the BERT model to use.
    - tokenizer (str): The name of the tokenizer to use.

    Returns:
    pd.DataFrame: The data with the extracted features.
    '''
    if data is None or bert_model is None or tokenizer is None:
        raise ValueError("All parameters must be specified.")

    # Load the BERT model and tokenizer
    bert_model = BertModel.from_pretrained(bert_model)
    tokenizer = BertTokenizer.from_pretrained(tokenizer)

    # Extract features from the eligibility criteria
    features = []
    for criteria in tqdm(data['Eligibility Criteria']):
        inputs = tokenizer(criteria, return_tensors='pt', truncation=True, max_length=512)
        outputs = bert_model(**inputs)
        features.append(outputs.pooler_output.detach().numpy())

    features = [feature.flatten() for feature in features]
    # Add the extracted features to the data
    features = pd.DataFrame(features)
    features.columns = [f"feature_{i}" for i in range(features.shape[1])]

    return features

In [75]:
def extract_features(data=None, bert_model=None, tokenizer=None, encoder=BinaryEncoder):
    '''
    Extracts features from the raw data using the BERT model and the specified encoder.

    Parameters:
    - data (pd.DataFrame): The raw data to extract features from.
    - bert_model (transformers.BertModel): The BERT model to use for feature extraction.
    - tokenizer (transformers.BertTokenizer): The tokenizer to use for tokenizing the input text.
    - encoder (category_encoders.Encoder): The encoder to use for encoding categorical features.

    Returns:
    pd.DataFrame: The extracted nct_id, features.
    '''
    if data is None or bert_model is None or tokenizer is None:
        raise ValueError("All arguments must be specified.")

    # Extract features from eligibility criteria using BERT
    criteria_features = extract_eligibility_features(data, bert_model, tokenizer)
    
    # Extract features from other fields using the specified encoder
    nominal_features = data.drop(columns=['Eligibility Criteria', 'NCT ID', 'Conditions'])
    
    encoder = encoder(cols=nominal_features.columns.tolist())
    other_features = encoder.fit_transform(nominal_features)

    # Combine the extracted features
    features = pd.concat([data['NCT ID'] ,criteria_features, other_features], axis=1)
    
    return features


In [76]:
feature_matrix = extract_features(data=raw_data, bert_model='microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract', tokenizer='microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract')

100%|██████████| 1266/1266 [1:17:28<00:00,  3.67s/it]


In [73]:
feature_matrix.to_csv(r'.\feature_matrix_biomedbert.csv', index=False)

In [ ]:
blue_bert = "bionlp/bluebert_pubmed_mimic_uncased_L-24_H-1024_A-16"

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler

# Set the range of cluster numbers to try
cluster_range = range(2, 11)  

# Initialize lists to store results
silhouette_scores = []
calinski_harabasz_scores = []
cross_val_scores = []

# Iterate over different cluster numbers
for n_clusters in cluster_range:
    # Initialize k-means model
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    
    # Fit the model
    kmeans.fit(feature_matrix.drop(columns=['NCT ID']))
    
    # Get cluster labels
    labels = kmeans.labels_
    
    # Calculate silhouette score
    silhouette_avg = silhouette_score(feature_matrix.drop(columns=['NCT ID']), labels)
    silhouette_scores.append(silhouette_avg)

    # Calculate Calinski-Harabasz score
    calinski_harabasz_score = calinski_harabasz_score(feature_matrix.drop(columns=['NCT ID']), labels)
    calinski_harabasz_scores.append(calinski_harabasz_score)
    
    # Perform cross-validation
    cv_scores = cross_val_score(kmeans, feature_matrix.drop(columns=['NCT ID']), cv=KFold(n_splits=5, shuffle=True, random_state=42))
    cross_val_scores.append(np.mean(cv_scores))

# Find the optimal number of clusters based on cross-validation scores
optimal_clusters = cluster_range[np.argmax(cross_val_scores)]

# Visualize the results (optional)
import matplotlib.pyplot as plt

plt.plot(cluster_range, silhouette_scores, label='Silhouette Score')
plt.plot(cluster_range, cross_val_scores, label='Cross-Validation Score')
plt.xlabel('Number of Clusters')
plt.ylabel('Score')
plt.legend()
plt.show()

print(f"Optimal Number of Clusters: {optimal_clusters}")


In [11]:
def train_model(algo='k-means', data=None):
    if algo == 'k-means':
        from sklearn.cluster import KMeans
        model = KMeans(n_clusters=4, random_state=42)
    elif algo == 'gmm':
        from sklearn.mixture import GaussianMixture
        model = GaussianMixture(n_components=4, random_state=42)
    
    cluster_labels = model.fit_predict(data.drop(columns=['NCT ID']))

    return model, cluster_labels

In [12]:
def evaluate_model(data, labels):
    from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, rand_score, adjusted_rand_score, mutual_info_score

    print(f"Silhouette score: {silhouette_score(data, labels)}")
    print(f"Calinski-Harabasz score: {calinski_harabasz_score(data, labels)}")
    print(f"Davies-Bouldin score: {davies_bouldin_score(data, labels)}")
    print(f"Rand score: {rand_score(data['Study Type'], labels)}")
    print(f"Adjusted Rand score: {adjusted_rand_score(data['Study Type'], labels)}")
    print(f"Mutual Information score: {mutual_info_score(data['Study Type'], labels)}")
